In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

MODELS = [("distilbert-base-uncased", "sst2"), ("bert-base-uncased", "sst2")]
SEED = 0                          # which fine-tuned checkpoint (speed does not depend on it)
K_BENCH = [12, 10, 8, 6, 4, 3, 2]  # heads kept PER LAYER (12 = dense)
ROUNDS = 3                        # repeated sweeps, config order rotated each round
LAT_N, LAT_WARM = 300, 30         # batch-1 latency: measured calls / warm-up calls (GPU)
CPU_LAT_N, CPU_LAT_WARM = 100, 10
THR_PASSES = 5                    # passes over the 872 val sentences for throughput (GPU)
CPU_THR_N = 256                   # sentences for CPU throughput (batch 32)
ENERGY_SECONDS = 15               # length of each energy window (GPU, batch 64)
GRID_KG_PER_KWH = 0.727           # India grid factor, CEA CO2 Baseline Database v20 (FY2023-24, weighted average).
                                  # SECONDARY-SOURCE VALUE: verify on cea.nic.in and cite the primary document.
OUT_DIR = "/kaggle/working/bench_results"
SMOKE = False                     # True: 1 model, K=[12,6], 1 round, short windows (about 3 min)
SEARCH = [
    "/kaggle/input/notebooks/*/*/ckpt/*/done.json",
    "/kaggle/input/*/ckpt/*/done.json",
    "/kaggle/input/**/done.json",
    "/kaggle/working/ckpt/**/done.json",
]
MAX_LEN = 128

import copy, gc, glob, json, platform, statistics, threading, time
import numpy as np
import pandas as pd

if SMOKE:
    MODELS, K_BENCH, ROUNDS = MODELS[:1], [12, 6], 1
    LAT_N, LAT_WARM, CPU_LAT_N, CPU_LAT_WARM, THR_PASSES, CPU_THR_N, ENERGY_SECONDS = 30, 5, 15, 3, 1, 64, 3
    OUT_DIR += "_smoke"


def flops_formula(D, Dff, dh, heads_per_layer, T=128):
    """Analytic FLOPs (multiply-add = 2 FLOPs) of the encoder for ONE sequence of T tokens.
    Returns (attention_module_flops, total_encoder_flops).  Embeddings, LayerNorm, classifier ignored."""
    attn, total = 0.0, 0.0
    for h in heads_per_layer:
        proj = 8.0 * T * D * h * dh                 # q,k,v (3 x 2*T*D*h*dh) + out projection (2*T*h*dh*D)
        mat = 4.0 * T * T * h * dh                  # QK^T and AV : 2 x (2*T*T*dh) per head
        ffn = 4.0 * T * D * Dff                     # two linear layers
        attn += proj + mat
        total += proj + mat + ffn
    return attn, total


def keep_uniform_by_score(score, k):
    L, H = score.shape
    keep = np.zeros((L, H), bool)
    for l in range(L):
        keep[l, np.argsort(-score[l], kind="stable")[:k]] = True
    return keep


def summarize(raw):
    """Median over rounds per (model, k); relative-to-dense columns."""
    num = raw.select_dtypes("number").columns.difference(["round", "k"])
    med = raw.groupby(["model", "k"])[list(num)].median().reset_index()
    out = []
    for m, g in med.groupby("model"):
        d = g[g.k == g.k.max()]
        if len(d) == 0:
            out.append(g); continue
        d = d.iloc[0]
        g = g.copy()
        g["gpu_lat_b1_speedup"] = d["gpu_lat_b1_ms_median"] / g["gpu_lat_b1_ms_median"]
        g["gpu_thr_b32_ratio"] = g["gpu_thr_b32_sps"] / d["gpu_thr_b32_sps"]
        g["gpu_thr_b128_ratio"] = g["gpu_thr_b128_sps"] / d["gpu_thr_b128_sps"]
        g["cpu_lat_b1_speedup"] = d["cpu_lat_b1_ms_median"] / g["cpu_lat_b1_ms_median"]
        g["cpu_thr_b32_ratio"] = g["cpu_thr_b32_sps"] / d["cpu_thr_b32_sps"]
        g["peak_mem_reduction_pct"] = 100 * (1 - g["gpu_peak_mem_mb"] / d["gpu_peak_mem_mb"])
        g["weight_mem_reduction_pct"] = 100 * (1 - g["gpu_weight_mb"] / d["gpu_weight_mb"])
        g["flops_total_rel"] = g["flops_total"] / d["flops_total"]
        g["flops_attn_rel"] = g["flops_attn"] / d["flops_attn"]
        g["energy_gross_rel"] = g["gpu_energy_j_per_1000"] / d["gpu_energy_j_per_1000"]
        out.append(g)
    return pd.concat(out)


import torch
import torch.nn as nn
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification

dev_ = torch.device("cuda" if torch.cuda.is_available() else "cpu")
HAVE_GPU = dev_.type == "cuda"


def get_attn(model):
    if hasattr(model, "distilbert"):
        ls = model.distilbert.transformer.layer
        return [(l.attention.q_lin, l.attention.k_lin, l.attention.v_lin, l.attention.out_lin) for l in ls], \
            model.config.n_heads, model.config.dim
    ls = model.bert.encoder.layer
    return [(l.attention.self.query, l.attention.self.key, l.attention.self.value, l.attention.output.dense) for l in ls], \
        model.config.num_attention_heads, model.config.hidden_size


def ffn_dim(model):
    c = model.config
    return getattr(c, "hidden_dim", None) or getattr(c, "intermediate_size")


def op_scores(model):
    """A-SICR score: spectral norm of each head's W_O slice (D x dh). CPU."""
    acc, H, D = get_attn(model)
    dh, L = D // H, len(acc)
    S = np.zeros((L, H))
    with torch.no_grad():
        for l, (_, _, _, o) in enumerate(acc):
            Wo = o.weight.detach().float().cpu()
            for h in range(H):
                S[l, h] = torch.linalg.svdvals(Wo[:, h * dh:(h + 1) * dh])[0].item()
    return S


def all_weight_scores(model):
    """frob, op, chain (beta_Q beta_K beta_V beta_O) for every head. CPU."""
    acc, H, D = get_attn(model)
    dh, L = D // H, len(acc)
    S = {k: np.zeros((L, H)) for k in ["frob", "op", "chain"]}
    with torch.no_grad():
        for l, (q, k, v, o) in enumerate(acc):
            Wq, Wk, Wv, Wo = [m.weight.detach().float().cpu() for m in (q, k, v, o)]
            for h in range(H):
                sl = slice(h * dh, (h + 1) * dh)
                sv = torch.linalg.svdvals(Wo[:, sl])
                S["op"][l, h] = sv[0].item()
                S["frob"][l, h] = torch.linalg.matrix_norm(Wo[:, sl], "fro").item()
                S["chain"][l, h] = (torch.linalg.matrix_norm(Wq[sl, :], 2) * torch.linalg.matrix_norm(Wk[sl, :], 2)
                                    * torch.linalg.matrix_norm(Wv[sl, :], 2)).item() * sv[0].item()
    return S


def zero_mask(model, keep):
    """Reference: zero the W_O columns of pruned heads (what the accuracy experiments did)."""
    acc, H, D = get_attn(model)
    dh = D // H
    with torch.no_grad():
        for l, (_, _, _, o) in enumerate(acc):
            for h in range(H):
                if not keep[l, h]:
                    o.weight[:, h * dh:(h + 1) * dh] = 0.0


def manual_prune(model, keep):
    """Physical removal by slicing q/k/v rows and W_O columns (mirrors what HF prune_heads does)."""
    acc, H, D = get_attn(model)
    dh = D // H
    is_distil = hasattr(model, "distilbert")
    layers = model.distilbert.transformer.layer if is_distil else model.bert.encoder.layer
    for l, (q, k, v, o) in enumerate(acc):
        idx = [h for h in range(H) if keep[l, h]]
        cols = torch.cat([torch.arange(h * dh, (h + 1) * dh) for h in idx])

        def rows(lin):
            n = nn.Linear(lin.in_features, len(cols), bias=lin.bias is not None)
            n.weight.data = lin.weight.data[cols].clone()
            if lin.bias is not None: n.bias.data = lin.bias.data[cols].clone()
            return n

        def colsl(lin):
            n = nn.Linear(len(cols), lin.out_features, bias=lin.bias is not None)
            n.weight.data = lin.weight.data[:, cols].clone()
            if lin.bias is not None: n.bias.data = lin.bias.data.clone()
            return n

        if is_distil:
            a = layers[l].attention
            a.q_lin, a.k_lin, a.v_lin, a.out_lin = rows(q), rows(k), rows(v), colsl(o)
            a.n_heads = len(idx); a.dim = len(idx) * dh
        else:
            sa = layers[l].attention.self
            sa.query, sa.key, sa.value = rows(q), rows(k), rows(v)
            layers[l].attention.output.dense = colsl(o)
            sa.num_attention_heads = len(idx); sa.all_head_size = len(idx) * dh


def build_physical(base, keep, check_batch=None):
    """Return (pruned model on CPU fp32, method used, max|logit diff| vs zero-masked reference)."""
    L, H = keep.shape
    heads = {l: [h for h in range(H) if not keep[l, h]] for l in range(L)}
    heads = {l: v for l, v in heads.items() if v}
    m, how = copy.deepcopy(base), "none"
    if heads:
        try:
            m.prune_heads(heads); how = "hf_prune_heads"
        except Exception as e:
            print(f"    HF prune_heads failed ({type(e).__name__}: {str(e)[:80]}) -> manual slicing")
            m = copy.deepcopy(base); manual_prune(m, keep); how = "manual"
    diff = np.nan
    if check_batch is not None:
        ref = copy.deepcopy(base); zero_mask(ref, keep)
        with torch.no_grad():
            try:
                diff = float((m.eval()(**check_batch).logits - ref.eval()(**check_batch).logits).abs().max())
            except Exception as e:
                print(f"    equivalence check raised {type(e).__name__}: {str(e)[:100]}")
                diff = float("inf")
        if diff > 1e-3 and how == "hf_prune_heads":
            print(f"    HF result differs from zeroing (max diff {diff:.2e}) -> trying manual slicing")
            m = copy.deepcopy(base); manual_prune(m, keep); how = "manual"
            with torch.no_grad():
                try:
                    diff = float((m.eval()(**check_batch).logits - ref.eval()(**check_batch).logits).abs().max())
                except Exception:
                    diff = float("inf")
        del ref
    return m.eval(), how, diff


def count_params(model):
    acc, H, D = get_attn(model)
    tot = sum(p.numel() for p in model.parameters())
    att = sum(sum(p.numel() for p in lin.parameters()) for tup in acc for lin in tup)
    return tot, att


def model_flops(model, T=128):
    acc, H, D = get_attn(model)
    dh = D // H
    hpl = [q.out_features // dh for (q, _, _, _) in acc]
    return flops_formula(D, ffn_dim(model), dh, hpl, T), hpl


def discover():
    runs = {}
    for pat in SEARCH:
        for p in glob.glob(pat, recursive=True):
            try: r = json.load(open(p))
            except Exception: continue
            runs[(r["model"], r["task"], r["seed"])] = os.path.dirname(p)
    return runs


def load_val(tok):
    ds = load_dataset("nyu-mll/glue", "sst2")
    val = ds["validation"]
    enc = tok(list(val["sentence"]), truncation=True, max_length=MAX_LEN)
    return enc["input_ids"], np.array(val["label"]), ds["train"]


def make_batches(tok, ids, bs, device, rng_seed=0, n_max=None):
    order = np.random.RandomState(rng_seed).permutation(len(ids))
    if n_max: order = order[:n_max]
    out = []
    for i in range(0, len(order), bs):
        feats = [{"input_ids": ids[j], "attention_mask": [1] * len(ids[j])} for j in order[i:i + bs]]
        b = tok.pad(feats, return_tensors="pt")
        out.append({k: v.to(device) for k, v in b.items()})
    return out


def fixed_batch(tok, ids, bs, length, device):
    feats = [{"input_ids": ids[i % len(ids)], "attention_mask": [1] * len(ids[i % len(ids)])} for i in range(bs)]
    b = tok.pad(feats, padding="max_length", max_length=length, return_tensors="pt")
    return {k: v.to(device) for k, v in b.items()}


def sync():
    if HAVE_GPU: torch.cuda.synchronize()


class EnergyMeter:
    def __init__(self):
        self.ok, self.mode, self.nv, self.h = False, None, None, None
        try:
            import pynvml
        except Exception:
            try:
                os.system("pip -q install nvidia-ml-py > /dev/null 2>&1")
                import pynvml
            except Exception as e:
                print("pynvml unavailable -> energy will be NaN:", e); return
        try:
            pynvml.nvmlInit()
            self.nv, self.h = pynvml, pynvml.nvmlDeviceGetHandleByIndex(0)
            try:
                pynvml.nvmlDeviceGetTotalEnergyConsumption(self.h); self.mode = "counter"
            except Exception:
                pynvml.nvmlDeviceGetPowerUsage(self.h); self.mode = "sampling"
            self.ok = True
            print("NVML energy mode:", self.mode, "| GPU:", pynvml.nvmlDeviceGetName(self.h))
        except Exception as e:
            print("NVML not usable -> energy will be NaN:", e)

    def idle_power_w(self, seconds=5.0):
        if not self.ok: return float("nan")
        xs, t0 = [], time.time()
        while time.time() - t0 < seconds:
            xs.append(self.nv.nvmlDeviceGetPowerUsage(self.h) / 1000.0); time.sleep(0.1)
        return float(np.median(xs))

    def start(self):
        if not self.ok: return
        if self.mode == "counter":
            self._e0 = self.nv.nvmlDeviceGetTotalEnergyConsumption(self.h)
        else:
            self._samples, self._stop = [], False
            def run():
                while not self._stop:
                    self._samples.append((time.time(), self.nv.nvmlDeviceGetPowerUsage(self.h) / 1000.0)); time.sleep(0.05)
            self._th = threading.Thread(target=run, daemon=True); self._th.start()

    def stop(self):
        if not self.ok: return float("nan")
        if self.mode == "counter":
            return (self.nv.nvmlDeviceGetTotalEnergyConsumption(self.h) - self._e0) / 1000.0     # mJ -> J
        self._stop = True; self._th.join()
        t = np.array([a for a, _ in self._samples]); p = np.array([b for _, b in self._samples])
        return float(((p[1:] + p[:-1]) / 2 * np.diff(t)).sum()) if len(t) > 1 else float("nan")


def lat_b1(model, singles, n, warm, gpu):
    ts = []
    with torch.inference_mode():
        for i in range(warm + n):
            b = singles[i % len(singles)]
            if gpu: torch.cuda.synchronize()
            t0 = time.perf_counter()
            model(**b)
            if gpu: torch.cuda.synchronize()
            if i >= warm: ts.append((time.perf_counter() - t0) * 1000)
    return float(np.median(ts)), float(np.percentile(ts, 95)), float(np.percentile(ts, 75) - np.percentile(ts, 25))


def throughput(model, batches, passes, gpu):
    nsent = sum(b["input_ids"].shape[0] for b in batches)
    rates = []
    with torch.inference_mode():
        for p in range(passes + 1):
            if gpu: torch.cuda.synchronize()
            t0 = time.perf_counter()
            for b in batches: model(**b)
            if gpu: torch.cuda.synchronize()
            if p > 0: rates.append(nsent / (time.perf_counter() - t0))        # pass 0 = warm-up
    return float(np.median(rates)), float(min(rates)), float(max(rates))


def val_acc(model, batches_pred, labels):
    preds = []
    with torch.inference_mode():
        for b in batches_pred: preds.append(model(**b).logits.argmax(1).cpu().numpy())
    return float((np.concatenate(preds) == labels[:sum(len(p) for p in preds)]).mean())


def measure_gpu(model_cpu, tok, ids, labels, meter, idle_w):
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.synchronize()
    mem0 = torch.cuda.memory_allocated()
    m = copy.deepcopy(model_cpu).to(dev_).half().eval()
    sync(); r = {"gpu_weight_mb": (torch.cuda.memory_allocated() - mem0) / 2 ** 20}
    # peak inference memory: batch 128 x 128 tokens
    fb = fixed_batch(tok, ids, 128, 128, dev_)
    torch.cuda.reset_peak_memory_stats()
    with torch.inference_mode(): m(**fb)
    sync(); r["gpu_peak_mem_mb"] = (torch.cuda.max_memory_allocated() - mem0) / 2 ** 20
    del fb
    singles = make_batches(tok, ids, 1, dev_, n_max=LAT_N + LAT_WARM)
    r["gpu_lat_b1_ms_median"], r["gpu_lat_b1_ms_p95"], r["gpu_lat_b1_ms_iqr"] = lat_b1(m, singles, LAT_N, LAT_WARM, True)
    del singles
    for bs in (32, 128):
        bt = make_batches(tok, ids, bs, dev_)
        med, lo, hi = throughput(m, bt, THR_PASSES, True)
        r[f"gpu_thr_b{bs}_sps"], r[f"gpu_thr_b{bs}_min"], r[f"gpu_thr_b{bs}_max"] = med, lo, hi
        del bt
    bt = make_batches(tok, ids, 128, dev_)
    r["val_acc_fp16"] = val_acc(m, bt, labels[np.random.RandomState(0).permutation(len(ids))])   # batches use the same permutation
    del bt
    # energy window (batch 64)
    if meter.ok:
        bt = make_batches(tok, ids, 64, dev_)
        with torch.inference_mode():
            for b in bt[:3]: m(**b)
            sync(); meter.start(); t0 = time.perf_counter(); n = 0
            while time.perf_counter() - t0 < ENERGY_SECONDS:
                for b in bt:
                    m(**b); n += b["input_ids"].shape[0]
                    if time.perf_counter() - t0 >= ENERGY_SECONDS: break
            sync(); dur = time.perf_counter() - t0; joules = meter.stop()
        r["gpu_energy_j_per_1000"] = joules / n * 1000
        r["gpu_energy_net_j_per_1000"] = (joules - idle_w * dur) / n * 1000
        r["gpu_avg_power_w"] = joules / dur
        r["co2_g_per_1M_inferences"] = (joules / n) * 1e6 / 3.6e6 * GRID_KG_PER_KWH * 1000
        del bt
    else:
        r.update(gpu_energy_j_per_1000=np.nan, gpu_energy_net_j_per_1000=np.nan, gpu_avg_power_w=np.nan, co2_g_per_1M_inferences=np.nan)
    del m
    gc.collect(); torch.cuda.empty_cache()
    return r


def measure_cpu(model_cpu, tok, ids):
    m = model_cpu.eval()
    singles = make_batches(tok, ids, 1, torch.device("cpu"), n_max=CPU_LAT_N + CPU_LAT_WARM)
    r = {}
    r["cpu_lat_b1_ms_median"], r["cpu_lat_b1_ms_p95"], r["cpu_lat_b1_ms_iqr"] = lat_b1(m, singles, CPU_LAT_N, CPU_LAT_WARM, False)
    bt = make_batches(tok, ids, 32, torch.device("cpu"), n_max=CPU_THR_N)
    r["cpu_thr_b32_sps"], r["cpu_thr_b32_min"], r["cpu_thr_b32_max"] = throughput(m, bt, 2, False)
    return r


def grad_importance(model, batches1, device):
    acc, H, D = get_attn(model)
    dh, L = D // H, len(acc)
    xi = torch.ones(L, H, device=device, requires_grad=True)
    hooks = []
    for l, (_, _, _, o) in enumerate(acc):
        def pre(mod, args, l=l):
            x = args[0]; B, T, Dm = x.shape
            return ((x.reshape(B, T, H, dh) * xi[l].view(1, 1, H, 1)).reshape(B, T, Dm),)
        hooks.append(o.register_forward_pre_hook(pre))
    imp = torch.zeros(L, H, device=device)
    model.eval()
    for b in batches1:
        loss = model(**b).loss
        g, = torch.autograd.grad(loss, xi)
        imp += g.abs()
    for h_ in hooks: h_.remove()
    return (imp / len(batches1)).detach().cpu().numpy()


@torch.no_grad()
def act_importance(model, batches, device):
    acc, H, D = get_attn(model)
    dh, L = D // H, len(acc)
    grams = []
    for (_, _, _, o) in acc:
        Wo = o.weight.detach().float().reshape(D, H, dh)
        grams.append(torch.einsum("ohd,ohe->hde", Wo, Wo))
    sums = torch.zeros(L, H, device=device)
    state = {"mask": None, "n": 0}
    hooks = []
    for l, (_, _, _, o) in enumerate(acc):
        def pre(mod, args, l=l):
            x = args[0].float().reshape(args[0].shape[0], args[0].shape[1], H, dh)
            q = torch.einsum("bthd,hde,bthe->bth", x, grams[l], x).clamp_min(0).sqrt()
            sums[l] += (q * state["mask"].unsqueeze(-1)).sum((0, 1))
        hooks.append(o.register_forward_pre_hook(pre))
    model.eval()
    for b in batches:
        state["mask"] = b["attention_mask"].float(); state["n"] += int(b["attention_mask"].sum().item())
        with torch.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
            model(**{k: v for k, v in b.items() if k != "labels"})
    for h_ in hooks: h_.remove()
    return (sums / max(state["n"], 1)).cpu().numpy()


def timed(fn, repeats=3):
    ts = []
    for _ in range(repeats):
        sync(); t0 = time.perf_counter(); fn(); sync(); ts.append(time.perf_counter() - t0)
    return float(np.median(ts)), float(min(ts)), float(max(ts))


def labelled_batches(tok, train, n, bs, device):
    sub = train.select(range(n))
    enc = tok(list(sub["sentence"]), truncation=True, max_length=MAX_LEN)
    out = []
    for i in range(0, n, bs):
        feats = [{"input_ids": enc["input_ids"][j], "attention_mask": [1] * len(enc["input_ids"][j])} for j in range(i, min(i + bs, n))]
        b = tok.pad(feats, return_tensors="pt")
        b["labels"] = torch.tensor(list(sub["label"][i:i + bs]))
        out.append({k: v.to(device) for k, v in b.items()})
    return out


def scoring_costs(base, tok, train, model_name):
    rows = []
    cpu = torch.device("cpu")

    def add(method, device, n, labels, backward, stats):
        rows.append(dict(model=model_name, method=method, device=device, n_examples=n, needs_labels=labels,
                         needs_backward=backward, seconds_median=stats[0], seconds_min=stats[1], seconds_max=stats[2]))
        print(f"    {method:22s} {device:4s} n={n:<4d} {stats[0]:8.3f}s")

    m_cpu = copy.deepcopy(base).eval()
    add("op_svd (A-SICR)", "cpu", 0, False, False, timed(lambda: op_scores(m_cpu), 5))
    add("all_weight_scores", "cpu", 0, False, False, timed(lambda: all_weight_scores(m_cpu), 3))
    for n in (8, 32, 128):
        b = labelled_batches(tok, train, n, 32, cpu)
        add("act", "cpu", n, False, False, timed(lambda: act_importance(m_cpu, b, cpu), 3))
    for n in (32, 128):
        b1 = labelled_batches(tok, train, n, 1, cpu)
        add("grad", "cpu", n, True, True, timed(lambda: grad_importance(m_cpu, b1, cpu), 2))
    # end to end on CPU: scores -> mask -> physical removal (k=6)
    def e2e_op():
        s = op_scores(m_cpu); keep = keep_uniform_by_score(s, 6); build_physical(m_cpu, keep, None)
    add("e2e_op_to_pruned_k6", "cpu", 0, False, False, timed(e2e_op, 3))
    b32 = labelled_batches(tok, train, 32, 32, cpu)
    def e2e_act():
        s = act_importance(m_cpu, b32, cpu); keep = keep_uniform_by_score(s, 6); build_physical(m_cpu, keep, None)
    add("e2e_act32_to_pruned_k6", "cpu", 32, False, False, timed(e2e_act, 3))
    if HAVE_GPU:
        m_gpu = copy.deepcopy(base).to(dev_).eval()
        for n in (8, 32, 128):
            b = labelled_batches(tok, train, n, 32, dev_)
            add("act", "gpu", n, False, False, timed(lambda: act_importance(m_gpu, b, dev_), 3))
        for n in (32, 128):
            b1 = labelled_batches(tok, train, n, 1, dev_)
            add("grad", "gpu", n, True, True, timed(lambda: grad_importance(m_gpu, b1, dev_), 3))
        del m_gpu
        gc.collect(); torch.cuda.empty_cache()
    return rows


def env_info():
    cpu = "?"
    try:
        for ln in open("/proc/cpuinfo"):
            if ln.startswith("model name"): cpu = ln.split(":")[1].strip(); break
    except Exception: pass
    import transformers
    info = dict(torch=torch.__version__, transformers=transformers.__version__, cuda=torch.version.cuda,
                gpu=torch.cuda.get_device_name(0) if HAVE_GPU else None, cpu=cpu, cpu_count=os.cpu_count(),
                torch_threads=torch.get_num_threads(), python=platform.python_version(),
                grid_kg_per_kwh=GRID_KG_PER_KWH, K_BENCH=K_BENCH, ROUNDS=ROUNDS, dtype_gpu="fp16", dtype_cpu="fp32",
                date=time.strftime("%Y-%m-%d %H:%M"))
    return info


def main():
    os.makedirs(OUT_DIR, exist_ok=True)
    runs = discover()
    meter = EnergyMeter() if HAVE_GPU else None
    idle_w = meter.idle_power_w(5.0) if (meter and meter.ok) else float("nan")
    print(f"GPU idle power (median of 5 s): {idle_w:.1f} W" if idle_w == idle_w else "GPU idle power: n/a")
    info = env_info(); info["gpu_idle_power_w"] = idle_w
    json.dump(info, open(f"{OUT_DIR}/env.json", "w"), indent=2)
    print(json.dumps(info, indent=1))
    raw_rows, score_rows = [], []
    for (mname, task) in MODELS:
        key = (mname, task, SEED)
        assert key in runs, f"checkpoint {key} not found. Seen: {sorted(runs)[:6]} ..."
        path = runs[key]
        tok = AutoTokenizer.from_pretrained(path)
        base = AutoModelForSequenceClassification.from_pretrained(path).eval()
        ids, labels, train = load_val(tok)
        acc, H, D = get_attn(base)
        L = len(acc)
        S = op_scores(base)
        check = tok.pad([{"input_ids": ids[i], "attention_mask": [1] * len(ids[i])} for i in range(16)], return_tensors="pt")
        print(f"\n######## {mname}/{task} seed {SEED}: L={L} H={H} D={D}")
        equiv = {}
        for r in range(ROUNDS):
            order = K_BENCH[r % len(K_BENCH):] + K_BENCH[:r % len(K_BENCH)]
            print(f"--- round {r + 1}/{ROUNDS}, order {order}")
            for k in order:
                keep = keep_uniform_by_score(S, k)
                pm, how, diff = build_physical(base, keep, check if r == 0 else None)
                if r == 0:
                    equiv[k] = (how, diff)
                    print(f"  k={k:2d}: physical removal via {how}; max|logit diff| vs zeroing = {diff:.2e}")
                    if not (diff <= 1e-3):
                        print("    !! equivalence check FAILED for this k -> skipped"); continue
                elif equiv.get(k, ("", 1))[1] > 1e-3:
                    continue
                (fa, ft), hpl = model_flops(pm, 128)
                tot, att = count_params(pm)
                row = dict(model=mname, task=task, round=r, k=k, heads_total=int(sum(hpl)), params_m=tot / 1e6,
                           attn_params_m=att / 1e6, flops_attn=fa, flops_total=ft, prune_method=how)
                row.update(measure_cpu(pm, tok, ids))
                if HAVE_GPU:
                    row.update(measure_gpu(pm, tok, ids, labels, meter, idle_w))
                raw_rows.append(row)
                print(f"  k={k:2d}: cpu b1 {row['cpu_lat_b1_ms_median']:.1f} ms | " +
                      (f"gpu b1 {row['gpu_lat_b1_ms_median']:.2f} ms | gpu thr128 {row['gpu_thr_b128_sps']:.0f}/s | "
                       f"peak {row['gpu_peak_mem_mb']:.0f} MB | E {row['gpu_energy_j_per_1000']:.1f} J/1000" if HAVE_GPU else ""))
                del pm; gc.collect()
            pd.DataFrame(raw_rows).to_csv(f"{OUT_DIR}/bench_raw.csv", index=False)
        print(f"\n--- scoring cost for {mname}")
        score_rows += scoring_costs(base, tok, train, mname)
        pd.DataFrame(score_rows).to_csv(f"{OUT_DIR}/scoring_costs.csv", index=False)
        json.dump({str(k): list(v) for k, v in equiv.items()}, open(f"{OUT_DIR}/equivalence_{mname}.json", "w"))
        del base; gc.collect()
    raw = pd.DataFrame(raw_rows)
    raw.to_csv(f"{OUT_DIR}/bench_raw.csv", index=False)
    summ = summarize(raw)
    summ.to_csv(f"{OUT_DIR}/bench_summary.csv", index=False)
    cols = ["model", "k", "heads_total", "params_m", "flops_total_rel", "flops_attn_rel", "gpu_lat_b1_speedup", "gpu_thr_b128_ratio",
            "cpu_lat_b1_speedup", "cpu_thr_b32_ratio", "weight_mem_reduction_pct", "peak_mem_reduction_pct", "energy_gross_rel", "val_acc_fp16"]
    print("\n===== SUMMARY (median over rounds) =====")
    print(summ[[c for c in cols if c in summ.columns]].round(3).to_string(index=False))
    print("\nALL DONE ->", OUT_DIR)


if __name__ == "__main__" or True:
    main()

NVML energy mode: counter | GPU: Tesla T4
GPU idle power (median of 5 s): 9.5 W
{
 "torch": "2.11.0+cu128",
 "transformers": "5.16.1",
 "cuda": "12.8",
 "gpu": "Tesla T4",
 "cpu": "Intel(R) Xeon(R) CPU @ 2.00GHz",
 "cpu_count": 4,
 "torch_threads": 2,
 "python": "3.13.15",
 "grid_kg_per_kwh": 0.727,
 "K_BENCH": [
  12,
  10,
  8,
  6,
  4,
  3,
  2
 ],
 "ROUNDS": 3,
 "dtype_gpu": "fp16",
 "dtype_cpu": "fp32",
 "date": "2026-10-29 13:37",
 "gpu_idle_power_w": 9.467
}


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

README.md:   0%|          | 0.00/35.3k [00:00<?, ?B/s]

sst2/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.11MB            

sst2/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

sst2/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 72.8kB            

... (5 outputs left out)

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]


######## distilbert-base-uncased/sst2 seed 0: L=6 H=12 D=768
--- round 1/3, order [12, 10, 8, 6, 4, 3, 2]
  k=12: physical removal via none; max|logit diff| vs zeroing = 0.00e+00
  k=12: cpu b1 32.3 ms | gpu b1 3.54 ms | gpu thr128 3894/s | peak 411 MB | E 17.5 J/1000
    HF prune_heads failed (AttributeError: 'DistilBertForSequenceClassification' object has no attribute 'prune_heads') -> manual slicing
  k=10: physical removal via manual; max|logit diff| vs zeroing = 7.15e-07
  k=10: cpu b1 36.0 ms | gpu b1 3.60 ms | gpu thr128 3986/s | peak 390 MB | E 17.1 J/1000
    HF prune_heads failed (AttributeError: 'DistilBertForSequenceClassification' object has no attribute 'prune_heads') -> manual slicing
  k= 8: physical removal via manual; max|logit diff| vs zeroing = 1.43e-06
  k= 8: cpu b1 33.9 ms | gpu b1 3.52 ms | gpu thr128 4142/s | peak 385 MB | E 16.6 J/1000
    HF prune_heads failed (AttributeError: 'DistilBertForSequenceClassification' object has no attribute 'prune_heads') -> m

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


######## bert-base-uncased/sst2 seed 0: L=12 H=12 D=768
--- round 1/3, order [12, 10, 8, 6, 4, 3, 2]
  k=12: physical removal via none; max|logit diff| vs zeroing = 0.00e+00
  k=12: cpu b1 65.4 ms | gpu b1 6.78 ms | gpu thr128 1764/s | peak 489 MB | E 37.4 J/1000
    HF prune_heads failed (AttributeError: 'BertForSequenceClassification' object has no attribute 'prune_heads') -> manual slicing
  k=10: physical removal via manual; max|logit diff| vs zeroing = 2.38e-06
  k=10: cpu b1 72.7 ms | gpu b1 6.95 ms | gpu thr128 1864/s | peak 466 MB | E 35.9 J/1000
    HF prune_heads failed (AttributeError: 'BertForSequenceClassification' object has no attribute 'prune_heads') -> manual slicing
  k= 8: physical removal via manual; max|logit diff| vs zeroing = 1.91e-06
  k= 8: cpu b1 68.6 ms | gpu b1 7.05 ms | gpu thr128 1984/s | peak 457 MB | E 34.5 J/1000
    HF prune_heads failed (AttributeError: 'BertForSequenceClassification' object has no attribute 'prune_heads') -> manual slicing
  k= 6: p